# LLM-Powered Medical Chatbot with RAG
## A Human-Centered Clinical Decision Support System

  
**Dataset:** HealthCareMagic 100k- 100k real doctor-patient dialogues  
**Approach:** Three-phase comparison- Original Baseline vs Improved Prompt vs RAG-Enhanced  
**Target Users:** ER Clinicians and Triage Staff  

## 1. Why -Problem Statement and Purpose

Getting quick and reliable medical information is not always easy, especially in critical care settings where clinicians operate under time pressure, high cognitive load, and constant interruptions.

### Motivation
Most AI medical chatbots rely solely on the LLM's general training knowledge. This often produces responses that are vague, overly cautious, or missing specific clinical details. Retrieval-Augmented Generation (RAG) offers a better approach, grounding the chatbot's responses in a real knowledge base of doctor-patient dialogues, 
so answers are informed by actual clinical experience rather than 
general knowledge alone.

As an **HCI researcher** with a focus on healthcare technologies, I am also interested in whether these AI responses are actually usable in critical care settings, not just technically correct, but clear, concise, and actionable for clinicians under pressure.

### Purpose
This project builds and evaluates a two-phase LLM-powered medical chatbot using the HealthCareMagic dataset. The goal is to compare 
a standard LLM baseline against a RAG-enhanced system, and evaluate both using technical metrics and a lightweight HCI rubric designed 
for critical care contexts.

### Expected Impact
- Help clinicians in critical care settings access relevant clinical information faster
- Show whether RAG meaningfully improves over a baseline LLM for medical question answering
- Introduce a human-centered evaluation layer for medical AI tools that goes beyond standard NLP metrics

## 2. What —Data, Techniques and Domain

### Dataset
This project uses the HealthCareMagic 100k dataset, a collection 
of real doctor-patient dialogues from the online medical consultation 
platform healthcaremagic.com, accessed freely via Hugging Face.:
https://huggingface.co/datasets/wangrongsheng/HealthCareMagic-100k-en

The full dataset contained 112,165 dialogues with no missing values. 
Each record contains the patient question, the doctor response, 
and a system instruction. Key findings from our EDA showed an 
average patient question length of 84 words and average doctor 
response length of 110 words, covering a wide range of medical 
topics and conditions.

A subset of 50,000 doctor responses was used as the RAG knowledge 
base for computational efficiency.

### Data Fields
The dataset contains three columns:
- input: the patient question —the primary input to the chatbot
- output: the doctor response —used as ground truth for evaluation and as the RAG knowledge base
- instruction: a consistent system prompt across all rows —used directly as our Phase 1a baseline prompt

### Techniques
Three core techniques were applied:

**Text Embeddings** using sentence-transformers (all-MiniLM-L6-v2) 
to convert doctor responses into numerical vectors that capture 
meaning rather than just keywords.

**Vector Search** using a FAISS flat index built from the 50,000 
embedded responses, enabling fast similarity search at query time.

**Retrieval-Augmented Generation (RAG)** connecting the retrieval 
system to GPT-4o-mini, so responses are grounded in real clinical 
dialogues rather than general training knowledge alone.

### Domain
This project sits at the intersection of NLP, Applied AI and HCI, 
with a focus on medical question answering and clinical decision 
support in critical care settings.

## 3. Who —Stakeholders and Use Cases

### Primary Users
The primary intended users of this system are clinicians in critical 
care settings, including doctors, nurses and triage staff who need 
fast access to relevant clinical information under time pressure. 
These users have little tolerance for long, verbose responses and 
need answers that are clear, specific and immediately actionable.

### Secondary Users
Secondary users include general patients seeking medical advice 
online, medical students looking for quick reference information, 
and healthcare educators using the system as a teaching tool.

### Why This Matters for HCI
As an HCI researcher focused on healthcare technologies, the design 
needs of clinicians in critical care settings are central to this 
project. These users operate under high cognitive load, frequent 
interruptions and significant time pressure. A chatbot that produces 
long, hedging or overly cautious responses is not just unhelpful 
in this context, it could be counterproductive.

This is why a lightweight HCI evaluation layer was added alongside 
the standard technical metrics, to assess whether the chatbot 
responses are actually suitable for these users and not just 
technically correct.

### Potential Use Cases
- Quick symptom assessment and differential diagnosis support
- Drug interaction and dosage reference during patient care
- Post-surgical complication triage guidance
- Medical student training and self-study support

## 4. How —Development Approach and Decisions

### Overview
The project was developed in Python using Jupyter Notebook and 
follows a three-phase pipeline designed to isolate the individual 
effects of prompt engineering and RAG retrieval.

### Step 1: Data Loading and EDA
The HealthCareMagic dataset was loaded from Hugging Face using 
the datasets library. EDA was performed to understand the structure 
and content of the dialogues, including length distributions, 
word frequency analysis and sample reviews.

### Step 2: Text Embeddings
50,000 doctor responses were converted into numerical vectors 
using sentence-transformers (all-MiniLM-L6-v2). This model was 
selected based on expert advice as a strong general-purpose 
embedding model that performs well on medical text.

Embeddings capture meaning rather than keywords, so similar 
medical concepts produce similar vectors even when different 
words are used.

### Step 3: Building the FAISS Index
The embeddings were stored in a FAISS flat index for fast 
similarity search. A flat index was chosen over more complex 
types because 50,000 vectors is well within the range where 
it is both accurate and efficient. The index was saved to disk 
after building so it only needed to be created once.

### Step 4: Retrieval Function
A retrieval function converts any patient question into a vector 
and searches the FAISS index for the top 3 most similar doctor 
responses. These form the context passed to the LLM in Phase 2.

### Step 5: Three-Phase Chatbot Development

**Phase 1a** uses the original system prompt from the dataset 
with no retrieval. This is the true baseline.

**Phase 1b** uses an improved system prompt instructing the model 
to be direct and specific. No retrieval. This isolates the effect 
of prompt engineering alone.

**Phase 2** uses the improved prompt plus the full RAG pipeline, 
retrieving the top 3 similar cases before generating a response.

### Key Design Decisions
The RAG pipeline was built manually using sentence-transformers, 
FAISS and the OpenAI API directly rather than LangChain, to keep 
the implementation transparent and explainable.

GPT-4o-mini was selected based on expert advice due to its strong 
performance, low cost and extensive documentation.

### Step 6: Evaluation
All three phases were tested on the same 20 randomly selected 
patient questions. Evaluation used LLM-as-Judge as the primary 
metric, manual scoring for validation, ROUGE as a secondary metric 
and a lightweight HCI rubric for critical care suitability.

## 5. How Good — Evaluation Results
Refer to end of notebook

## 6. Where — Deployment and Chat Interface 
Refer to end of notebook

## 7. Experiences — Reflection and Lessons Learned
Refer to end of notebook

In [ ]:
import subprocess
packages = [
    "openai",
    "langchain",
    "langchain-openai", 
    "langchain-community",
    "sentence-transformers",
    "faiss-cpu",
    "datasets",
    "pandas",
    "seaborn",
    "matplotlib",
    "nltk",
    "rouge-score",
    "ipywidgets"
]
for package in packages:
    subprocess.run(["pip", "install", package])
print("✅ All packages installed!")

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import nltk
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')

print("✅ All libraries imported successfully!")

In [ ]:
from openai import OpenAI
import os

# store API key as environment variable - never hardcode it
if not os.environ.get("CHATBOT_API_KEY"):
    raise ValueError("Set CHATBOT_API_KEY in your environment before running this notebook.")
client = OpenAI(api_key=os.environ["CHATBOT_API_KEY"])

print("OpenAI client ready!")

## Section 1: Loading Dataset

In [ ]:

from datasets import load_dataset

print("Loading HealthCareMagic 100k dataset from Hugging Face...")
dataset = load_dataset("wangrongsheng/HealthCareMagic-100k-en")

# Convert to pandas dataframe
df = pd.DataFrame(dataset['train'])

print(f"✅ Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst row sample:")
df.head(2)

In [ ]:
import urllib.request
try:
    urllib.request.urlopen('https://huggingface.co', timeout=5)
    print("Hugging Face is reachable")
except:
    print("Cannot reach Hugging Face — check your internet connection")

In [ ]:
# QUICK DATA PEEK 
print("=== SAMPLE PATIENT QUESTION ===")
print(df.iloc[0]['input'])
print("\n=== SAMPLE DOCTOR RESPONSE ===")
print(df.iloc[0]['output'])

## Section 2: Exploratory Data Analysis (EDA)
Understanding the structure and content of the HealthCareMagic dataset
before building the chatbot system.

In [ ]:
# Getting a feel for the dataset first
print("Total dialogues:", len(df))
print("Columns:", df.columns.tolist())
print("Any missing values?")
print(df.isnull().sum())

# How long are the (columns) questions and responses are on average?
df['question_length'] = df['input'].apply(lambda x: len(x.split()))
df['response_length'] = df['output'].apply(lambda x: len(x.split()))

print("\nPatient Question Length (words):")
print(df['question_length'].describe().round(2))

print("\nDoctor Response Length (words):")
print(df['response_length'].describe().round(2))

* Average patient question is 84 words 
* Average doctor response is 110 words 
* No missing values at all — clean dataset
* One outlier question at 2,218 words 

In [ ]:
# ── VISUALIZATIONS 

from collections import Counter
import re

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('HealthCareMagic Dataset - EDA', fontsize=16, fontweight='bold')

# Plot 1 — How long are patient questions?
axes[0,0].hist(df['question_length'], bins=50, color='steelblue', edgecolor='white')
axes[0,0].set_title('Patient Question Length')
axes[0,0].set_xlabel('Word Count')
axes[0,0].set_ylabel('Frequency')
axes[0,0].axvline(df['question_length'].mean(), color='red',
                   linestyle='--', label=f"Mean: {df['question_length'].mean():.0f} words")
axes[0,0].legend()

# Plot 2 — How long are doctor responses? 
axes[0,1].hist(df['response_length'], bins=50, color='seagreen', edgecolor='white')
axes[0,1].set_title('Doctor Response Length')
axes[0,1].set_xlabel('Word Count')
axes[0,1].set_ylabel('Frequency')
axes[0,1].axvline(df['response_length'].mean(), color='red',
                   linestyle='--', label=f"Mean: {df['response_length'].mean():.0f} words")
axes[0,1].legend()

# Plot 3 — Do longer questions get longer answers? Question length vs response length scatter plot
axes[1,0].scatter(df['question_length'], df['response_length'],
                   alpha=0.3, color='purple', s=10)
axes[1,0].set_title('Question Length vs Response Length')
axes[1,0].set_xlabel('Question Word Count')
axes[1,0].set_ylabel('Response Word Count')

# Plot 4 — What do patients talk about most? Top 20 words in questions
all_words = ' '.join(df['input'].tolist()).lower()
all_words = re.sub(r'[^a-z\s]', '', all_words)
word_freq = Counter(all_words.split())

# Remove common stopwords and short words
stopwords = {'i', 'the', 'a', 'and', 'to', 'of', 'in', 'is', 'my',
             'for', 'have', 'been', 'with', 'are', 'it', 'me', 'that',
             'was', 'he', 'she', 'this', 'but', 'on', 'at', 'by', 'an',
             'be', 'as', 'or', 'do', 'not', 'so', 'if', 'we', 'his',
             'her', 'from', 'had', 'can', 'also', 'any', 'your', 'you'}

filtered = {w: c for w, c in word_freq.items()
            if w not in stopwords and len(w) > 2}
top_words = dict(sorted(filtered.items(),
                 key=lambda x: x[1], reverse=True)[:20])

axes[1,1].barh(list(top_words.keys())[::-1],
               list(top_words.values())[::-1], color='coral')
axes[1,1].set_title('Top 20 Words in Patient Questions')
axes[1,1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Done! EDA plots saved.")

In [ ]:
# what is the instruction column?
print(df['instruction'].iloc[0])
print("\nUnique instruction values:", df['instruction'].nunique())

Sample Dialogues 

In [ ]:
# few random examples to understand the data better
print("=== 3 RANDOM PATIENT-DOCTOR DIALOGUES ===\n")

samples = df.sample(3, random_state=42)

for i, (_, row) in enumerate(samples.iterrows(), 1):
    print(f"--- Example {i} ---")
    print(f"PATIENT: {row['input'][:300]}...")
    print(f"DOCTOR:  {row['output'][:300]}...")
    print()

## EDA Summary

The HealthCareMagic dataset contains **112,165 real doctor-patient dialogues** 
with no missing values, making it clean and ready for use.

Key findings:
- Average patient question: **84 words** — detailed symptom descriptions
- Average doctor response: **110 words** — concise but thorough answers
- Topics are highly varied: from neurological symptoms to kidney disease to sinusitis
- The `instruction` column contains a consistent system prompt used across all dialogues:
  *"If you are a doctor, please answer the medical questions based on the patient's description."*
- This instruction will be used directly as the system prompt in our chatbot

My conclusion is that the dataset is well suited for building a RAG-based medical chatbot knowledge base.

## Section 3: Sentence-transformers for embeddings (Building the knowledge base)

Before building the chatbot, we need to prepare our knowledge base.
This involves two steps:
1. Converting doctor responses into numerical vectors (embeddings)
2. Storing those vectors in a FAISS index for fast similarity search

In [ ]:
from sentence_transformers import SentenceTransformer

# load the embedding model
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

# we use the doctor responses as our knowledge base
print("\nPreparing knowledge base...")
knowledge_base = df.head(50000).reset_index(drop=True)
doctor_responses = knowledge_base['output'].tolist()
print(f"Knowledge base size: {len(doctor_responses):,} doctor responses")

# convert doctor responses into vectors
print("\nEmbedding doctor responses...")
print("This may take a few minutes...\n")
embeddings = embedder.encode(
    doctor_responses,
    show_progress_bar=True,
    batch_size=64
)

# save to disk so we don't have to redo this
import numpy as np
np.save('doctor_embeddings.npy', embeddings)

print(f"\nEmbeddings shape: {embeddings.shape}")
print("Done! Embeddings saved to disk.")

## Section 4: Building the FAISS Index for vector search

FAISS (Facebook AI Similarity Search) is a library that lets us search 
through thousands of vectors instantly. We store all doctor response 
vectors in a FAISS index so that at query time, we can find the most 
similar cases in milliseconds.

In [ ]:
import faiss

# build the FAISS flat index
# flat index is best for our size — simple and accurate
print("Building FAISS index...")
dimension = embeddings.shape[1]  # 384 dimensions
index = faiss.IndexFlatL2(dimension)

# add all embeddings to the index
index.add(embeddings.astype('float32'))

# save to disk
faiss.write_index(index, 'medical_faiss.index')

print(f"FAISS index built!")
print(f"Total vectors indexed: {index.ntotal:,}")
print("Saved to medical_faiss.index")

## Section 5: Retrieval Function

This function takes a patient question, converts it into a vector,
and searches the FAISS index for the most similar doctor responses.
This is the retrieval step in our RAG pipeline.

In [ ]:
def retrieve_similar_cases(query, top_k=3):
    
    # step 1 - convert patient question to a vector
    query_vector = embedder.encode([query]).astype('float32')
    
    # step 2 - search FAISS for most similar doctor responses
    distances, indices = index.search(query_vector, top_k)
    
    # step 3 - return the matching cases
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'patient_question': knowledge_base['input'].iloc[idx],
            'doctor_response': knowledge_base['output'].iloc[idx],
            'similarity_score': float(distances[0][i])
        })
    return results

# test the retrieval function
print("Testing retrieval on: 'I have chest pain and shortness of breath'\n")
test_cases = retrieve_similar_cases("I have chest pain and shortness of breath")

for i, case in enumerate(test_cases, 1):
    print(f"--- Retrieved Case {i} ---")
    print(f"Patient: {case['patient_question'][:150]}...")
    print(f"Doctor:  {case['doctor_response'][:150]}...")
    print(f"Distance score: {case['similarity_score']:.4f}")
    print()

## Section 6: Chatbot Phases

We build and compare three phases:

- **Phase 1a** — Original system prompt, no RAG (true baseline)
- **Phase 1b** — Improved system prompt, no RAG (prompt engineering only)
- **Phase 2**  — Improved system prompt + RAG (full system)

This three-way comparison lets us isolate the effect of prompt 
engineering separately from the effect of RAG retrieval.

### System Prompts

In [ ]:
# original prompt — taken directly from the dataset
SYSTEM_PROMPT_ORIGINAL = df['instruction'].iloc[0]

# improved prompt — developed during testing
SYSTEM_PROMPT_IMPROVED = """You are an experienced medical doctor. 
Answer the patient's medical question with a specific diagnosis 
and clear recommendations where possible. 
Be direct, concise and helpful."""

print("Original prompt:")
print(SYSTEM_PROMPT_ORIGINAL)
print("\nImproved prompt:")
print(SYSTEM_PROMPT_IMPROVED)

### Phase 1a: Baseline Chatbot using OpenAI — Original Prompt, No RAG

The patient question is sent directly to GPT-4o-mini using the 
original system prompt from the dataset. No retrieval, no context.
This is our true baseline.

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=os.environ["CHATBOT_API_KEY"])

def phase1a_chatbot(patient_question):
    
    # send question directly to LLM with original prompt
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_ORIGINAL},
            {"role": "user", "content": patient_question}
        ],
        max_tokens=300,
        temperature=0.3
    )
    return response.choices[0].message.content

# quick test
print("Phase 1a test response:")
print(phase1a_chatbot(df['input'].iloc[0]))

### Phase 1b: Improved Prompt Chatbot — No RAG

Same as Phase 1a but with an improved system prompt.
This isolates the effect of prompt engineering alone,
without any retrieval augmentation.

In [ ]:
def phase1b_chatbot(patient_question):
    
    # send question directly to LLM with improved prompt
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_IMPROVED},
            {"role": "user", "content": patient_question}
        ],
        max_tokens=300,
        temperature=0.3
    )
    return response.choices[0].message.content

# quick test
print("Phase 1b test response:")
print(phase1b_chatbot(df['input'].iloc[0]))

### Phase 2: RAG-Enhanced Chatbot — Improved Prompt + Retrieval

This is the full RAG system. Before sending the question to the LLM,
we first retrieve the 3 most similar doctor-patient cases from our 
FAISS knowledge base. These are added as context to the prompt,
grounding the LLM response in real clinical dialogue.

In [ ]:
def phase2_rag_chatbot(patient_question, top_k=3):
    
    # step 1 - retrieve similar cases from knowledge base
    similar_cases = retrieve_similar_cases(patient_question, top_k)
    
    # step 2 - build context string from retrieved cases
    context = ""
    for i, case in enumerate(similar_cases, 1):
        context += f"Case {i}:\n"
        context += f"Patient: {case['patient_question']}\n"
        context += f"Doctor: {case['doctor_response']}\n\n"
    
    # step 3 - build augmented prompt with retrieved context
    augmented_prompt = f"""Here are {top_k} similar cases from real 
doctor-patient consultations:

{context}
Based on these similar cases and your medical knowledge, 
answer the following patient question:

{patient_question}"""

    # step 4 - send augmented prompt to LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_IMPROVED},
            {"role": "user", "content": augmented_prompt}
        ],
        max_tokens=300,
        temperature=0.3
    )
    return response.choices[0].message.content

# quick test
print("Phase 2 RAG test response:")
print(phase2_rag_chatbot(df['input'].iloc[0]))

### Running All Three Phases on 20 Test Questions

We now run all three phases on the same 20 randomly selected 
patient questions so we can compare results fairly.

In [ ]:
import time

# select 20 random questions
# random_state=42 ensures we always get the same 20 questions
print("Selecting 20 test questions...")
test_set = df.sample(20, random_state=42).reset_index(drop=True)
print(f"Done! {len(test_set)} questions selected.\n")

# empty lists to store responses
phase1a_responses = []
phase1b_responses = []
phase2_responses  = []

print("Running all three phases...")
print("This will take about 5-6 minutes.\n")

for i, row in test_set.iterrows():
    
    # phase 1a - original prompt, no RAG
    r1a = phase1a_chatbot(row['input'])
    phase1a_responses.append(r1a)
    
    # phase 1b - improved prompt, no RAG
    r1b = phase1b_chatbot(row['input'])
    phase1b_responses.append(r1b)
    
    # phase 2 - improved prompt + RAG
    r2 = phase2_rag_chatbot(row['input'])
    phase2_responses.append(r2)
    
    print(f"✓ Question {i+1}/20 complete")
    time.sleep(1)

# save all responses to test_set dataframe
print("\nSaving all responses...")
test_set['phase1a_response'] = phase1a_responses
test_set['phase1b_response'] = phase1b_responses
test_set['phase2_response']  = phase2_responses

print("Done! Showing sample output for Question 1:\n")
print(f"PATIENT:  {test_set['input'].iloc[0][:200]}...")
print(f"\nPhase 1a: {test_set['phase1a_response'].iloc[0][:200]}...")
print(f"\nPhase 1b: {test_set['phase1b_response'].iloc[0][:200]}...")
print(f"\nPhase 2:  {test_set['phase2_response'].iloc[0][:200]}...")
print(f"\nActual:   {test_set['output'].iloc[0][:200]}...")

* Phase 1a — says "I'm not a doctor" despite having a system prompt — weakest response
* Phase 1b — gives a specific diagnosis "nocturnal leg..." — most specific
* Phase 2 — gives a different diagnosis "restless legs syndrome" — grounded but different from actual
* Actual doctor — lists multiple possibilities including Phlebitis — most comprehensive

In [ ]:
# save test_set to disk before evaluation
# this way if kernel restarts we don't lose our 20 responses
test_set.to_csv('test_set_responses.csv', index=False)
print("Test set saved to test_set_responses.csv!")
print(f"Shape: {test_set.shape}")
print(f"Columns: {test_set.columns.tolist()}")

In [ ]:
''' # quick reload if kernel restarts
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = np.load('doctor_embeddings.npy')
index = faiss.read_index('medical_faiss.index')
knowledge_base = df.head(50000).reset_index(drop=True)
print(f"Reloaded! Vectors in index: {index.ntotal:,}")'''

## Section 7: Evaluation

We evaluate all three phases using three methods:

1. **Manual Scoring** — we personally score 5 responses to validate 
   the LLM judge
2. **LLM-as-Judge** — GPT-4o-mini scores all 20 responses on 
   correctness, relevance, clarity and actionability
3. **ROUGE Scores** — measures text overlap as a secondary metric

Note: LLM-as-Judge is our primary evaluation method as ROUGE alone 
is not sufficient for open-ended medical responses.

In [ ]:
# manually review first 5 questions before running LLM judge
# score each response 1-3 on each criterion
# 1 = Poor, 2 = Acceptable, 3 = Good

print("="*60)
for i in range(5):
    print(f"\nQUESTION {i+1}:")
    print(test_set['input'].iloc[i][:200])
    print(f"\nPhase 1a: {test_set['phase1a_response'].iloc[i]}")
    print(f"\nPhase 1b: {test_set['phase1b_response'].iloc[i]}")
    print(f"\nPhase 2:  {test_set['phase2_response'].iloc[i]}")
    print(f"\nActual:   {test_set['output'].iloc[i]}")
    print("\n" + "="*60)

### LLM Judge Function

In [ ]:
# define LLM judge function
def llm_judge(patient_question, doctor_reference, chatbot_response, phase_name):
    
    judge_prompt = f"""You are a medical expert evaluating an AI chatbot response.

Compare the chatbot response to the actual doctor's answer and score it.
Return ONLY a JSON object, nothing else. No extra text.

Patient Question:
{patient_question}

Actual Doctor Response:
{doctor_reference}

Chatbot Response:
{chatbot_response}

Score each criterion from 1 to 3:
1 = Poor
2 = Acceptable  
3 = Good

Return exactly this format:
{{
  "correctness": <1-3>,
  "relevance": <1-3>,
  "clarity": <1-3>,
  "actionability": <1-3>,
  "reasoning": "<one sentence explanation>"
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": judge_prompt}],
        max_tokens=200,
        temperature=0
    )
    
    try:
        result = json.loads(response.choices[0].message.content)
        result['phase'] = phase_name
        result['total'] = (result['correctness'] + result['relevance'] + 
                          result['clarity'] + result['actionability'])
        return result
    except:
        print(f"Parsing error for {phase_name}")
        return {'phase': phase_name, 'correctness': 0, 'relevance': 0,
                'clarity': 0, 'actionability': 0, 'total': 0,
                'reasoning': 'parsing error'}

print("LLM judge function ready!")

In [ ]:
import json
import time

# run LLM judge on first 5 questions
print("Validating LLM judge on first 5 questions...\n")

validation_results = []

for i in range(5):
    row = test_set.iloc[i]
    
    r1a = llm_judge(row['input'], row['output'],  # pyright: ignore[reportUndefinedVariable]
                    row['phase1a_response'], 'Phase 1a')
    r1b = llm_judge(row['input'], row['output'], 
                    row['phase1b_response'], 'Phase 1b')
    r2  = llm_judge(row['input'], row['output'], 
                    row['phase2_response'],  'Phase 2')
    
    validation_results.extend([r1a, r1b, r2])
    print(f"✓ Question {i+1}/5 evaluated")
    print(f"  1a: {r1a}")
    print(f"  1b: {r1b}")
    print(f"  2:  {r2}\n")
    time.sleep(1)

# create dataframe
val_df = pd.DataFrame(validation_results)
print("\nDone! LLM Judge summary:")
print(val_df.groupby('phase')[['correctness', 'relevance',
                                'clarity', 'actionability',
                                'total']].mean().round(2))

### Compare LLM Judge vs Manual Scores

In [ ]:
# our manual scores from earlier
manual_scores = {
    'Phase 1a': {'correctness': 2.0, 'relevance': 2.4, 
                 'clarity': 2.2, 'actionability': 2.0, 'total': 8.6},
    'Phase 1b': {'correctness': 2.2, 'relevance': 2.8, 
                 'clarity': 3.0, 'actionability': 2.8, 'total': 10.8},
    'Phase 2':  {'correctness': 2.6, 'relevance': 3.0, 
                 'clarity': 2.8, 'actionability': 2.8, 'total': 11.2}
}

manual_df = pd.DataFrame(manual_scores).T
manual_df.index.name = 'phase'

llm_val = val_df.groupby('phase')[['correctness', 'relevance',
                                    'clarity', 'actionability',
                                    'total']].mean().round(2)

print("="*50)
print("MANUAL SCORES (our scoring):")
print(manual_df.to_string())
print("\nLLM JUDGE SCORES (first 5 questions):")
print(llm_val.to_string())
print("="*50)
print("\nIf scores are broadly similar, LLM judge is validated!")
print("We can then run it on all 20 questions with confidence.")

In [ ]:
# confirmed - LLM judge validated, running on all 20 questions
print("LLM Judge validated!")
print("Direction of scores agrees with manual scoring.")
print("Ready to run on all 20 questions.")

### Run LLM Judge on All 20 Questions

In [ ]:
import time

print("Running LLM judge on all 20 questions...")
print("60 evaluations total (20 questions x 3 phases)\n")

all_judge_results = []

for i, row in test_set.iterrows():
    
    # judge all three phases for this question
    r1a = llm_judge(row['input'], row['output'],
                    row['phase1a_response'], 'Phase 1a')
    r1b = llm_judge(row['input'], row['output'],
                    row['phase1b_response'], 'Phase 1b')
    r2  = llm_judge(row['input'], row['output'],
                    row['phase2_response'],  'Phase 2')
    
    all_judge_results.extend([r1a, r1b, r2])
    print(f"✓ Question {i+1}/20 evaluated")
    time.sleep(1)

# save results to dataframe
judge_df = pd.DataFrame(all_judge_results)

# save to disk
judge_df.to_csv('judge_results.csv', index=False)

print("\nDone! Full LLM judge results:")
print(judge_df.groupby('phase')[['correctness', 'relevance',
                                  'clarity', 'actionability',
                                  'total']].mean().round(2))

### Visualize LLM Judge Results

## Understanding the Three Phases

To fairly evaluate the impact of both prompt engineering and RAG, 
we tested three phases using the same 20 patient questions:

**Phase 1a — True Baseline (Original Prompt, No RAG)**
- Uses the original system prompt taken directly from the dataset:
  *"If you are a doctor, please answer the medical questions 
  based on the patient's description."*
- No retrieval — patient question goes directly to GPT-4o-mini
- This is our true baseline — no modifications whatsoever
- Represents what the LLM knows from general training alone

**Phase 1b — Prompt Engineering Only (Improved Prompt, No RAG)**
- Uses an improved system prompt we developed during testing:
  *"You are an experienced medical doctor. Answer the patient's 
  medical question with a specific diagnosis and clear 
  recommendations where possible. Be direct, concise and helpful."*
- No retrieval — patient question still goes directly to GPT-4o-mini
- Isolates the effect of prompt engineering alone
- Shows how much improvement we can get without any RAG

**Phase 2 — Full RAG System (Improved Prompt + Retrieval)**
- Uses the same improved system prompt as Phase 1b
- Adds retrieval — top 3 most similar doctor-patient cases are 
  fetched from our FAISS knowledge base of 50,000 dialogues
- Retrieved cases are added as context before GPT-4o-mini answers
- This is our full system — prompt engineering + RAG combined
- Shows whether grounding responses in real clinical data 
  improves performance beyond prompt engineering alone

**Why three phases?**
Comparing all three lets us isolate two effects separately:
- Phase 1a → Phase 1b shows the impact of prompt engineering alone
- Phase 1b → Phase 2 shows the impact of adding RAG on top

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# summary data
summary = judge_df.groupby('phase')[['correctness', 'relevance',
                                      'clarity', 'actionability',
                                      'total']].mean().round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LLM Judge Evaluation Results — All 20 Questions', 
             fontsize=14, fontweight='bold')

# plot 1 - scores by criterion
criteria = ['correctness', 'relevance', 'clarity', 'actionability']
x = np.arange(len(criteria))
width = 0.25

axes[0].bar(x - width, summary.loc['Phase 1a', criteria], 
            width, label='Phase 1a', color='steelblue')
axes[0].bar(x,         summary.loc['Phase 1b', criteria], 
            width, label='Phase 1b', color='seagreen')
axes[0].bar(x + width, summary.loc['Phase 2',  criteria], 
            width, label='Phase 2',  color='coral')

axes[0].set_title('Average Score by Criterion')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Correctness', 'Relevance', 
                          'Clarity', 'Actionability'])
axes[0].set_ylabel('Average Score (1-3)')
axes[0].set_ylim(2, 3.2)
axes[0].legend()
axes[0].set_yticks([2.0, 2.2, 2.4, 2.6, 2.8, 3.0])

# plot 2 - total scores
phases = ['Phase 1a', 'Phase 1b', 'Phase 2']
totals = [summary.loc[p, 'total'] for p in phases]
colors = ['steelblue', 'seagreen', 'coral']

bars = axes[1].bar(phases, totals, color=colors, width=0.5)
axes[1].set_title('Total Score by Phase (max 12)')
axes[1].set_ylabel('Average Total Score')
axes[1].set_ylim(10, 12)

# add score labels on bars
for bar, total in zip(bars, totals):
    axes[1].text(bar.get_x() + bar.get_width()/2, 
                  bar.get_height() + 0.05,
                  f'{total:.2f}', ha='center', 
                  fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('llm_judge_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualization saved!")

## ROUGE Scores (Secondary Metric)

ROUGE measures text overlap between chatbot responses and actual 
doctor answers. This is used as a 
secondary metric only — since ROUGE measures word similarity, 
not medical correctness or helpfulness.

In [ ]:
from rouge_score import rouge_scorer

# initialize scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], 
                                   use_stemmer=True)

def compute_rouge(predictions, references):
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rl.append(scores['rougeL'].fmeasure)
    return {
        'ROUGE-1': round(np.mean(r1), 4),
        'ROUGE-2': round(np.mean(r2), 4),
        'ROUGE-L': round(np.mean(rl), 4)
    }

# compute for all three phases
print("Computing ROUGE scores...\n")
rouge_1a = compute_rouge(test_set['phase1a_response'], test_set['output'])
rouge_1b = compute_rouge(test_set['phase1b_response'], test_set['output'])
rouge_2  = compute_rouge(test_set['phase2_response'],  test_set['output'])

# display as dataframe
rouge_df = pd.DataFrame({
    'Phase': ['Phase 1a (original, no RAG)',
              'Phase 1b (improved, no RAG)',
              'Phase 2  (improved + RAG)'],
    'ROUGE-1': [rouge_1a['ROUGE-1'], rouge_1b['ROUGE-1'], rouge_2['ROUGE-1']],
    'ROUGE-2': [rouge_1a['ROUGE-2'], rouge_1b['ROUGE-2'], rouge_2['ROUGE-2']],
    'ROUGE-L': [rouge_1a['ROUGE-L'], rouge_1b['ROUGE-L'], rouge_2['ROUGE-L']]
})

print("ROUGE Scores:")
print(rouge_df.to_string(index=False))

## HCI Evaluation (Secondary Layer)

As a secondary evaluation layer, we assess chatbot responses through 
a human-centered lens informed by HCI research on designing technologies 
for critical care settings.

We evaluate 10 responses manually using a simple rubric based on:
- Situation Awareness Theory (Endsley) 
- Cognitive Load Theory

The rubric assesses whether responses are suitable for clinicians 
in critical care settings where time pressure and cognitive load

In [ ]:
# HCI evaluation rubric
# manually score 10 responses across all three phases
# criteria designed for critical care settings

hci_criteria = {
    'clarity':       'Is the response easy to understand under time pressure?',
    'conciseness':   'Is the response brief enough for a clinical setting?',
    'actionability': 'Does it tell the clinician what to DO next?',
    'accuracy':      'Does it align with the actual doctor response?',
    'cognitive_load': 'Does it avoid overwhelming with unnecessary detail?'
}

print("HCI Evaluation Criteria:")
print("="*60)
for criterion, description in hci_criteria.items():
    print(f"{criterion.upper()}: {description}")
print("="*60)
print("\nNow printing 10 responses for manual HCI scoring...\n")

for i in range(10):
    print(f"\n{'='*60}")
    print(f"QUESTION {i+1}:")
    print(test_set['input'].iloc[i][:150])
    print(f"\nPhase 1a: {test_set['phase1a_response'].iloc[i][:150]}")
    print(f"\nPhase 1b: {test_set['phase1b_response'].iloc[i][:150]}")
    print(f"\nPhase 2:  {test_set['phase2_response'].iloc[i][:150]}")
    print(f"\nActual:   {test_set['output'].iloc[i][:150]}")

### HCI Scores Input

In [ ]:
# HCI rubric scores based on manual evaluation of 10 responses
# criteria designed for critical care settings
# scoring: 1=Poor, 2=Acceptable, 3=Good

hci_scores = {
    'Phase 1a': {
        'clarity':        2.5,
        'conciseness':    1.9,
        'actionability':  2.1,
        'accuracy':       1.7,
        'cognitive_load': 1.9
    },
    'Phase 1b': {
        'clarity':        2.9,
        'conciseness':    2.0,
        'actionability':  2.8,
        'accuracy':       2.0,
        'cognitive_load': 2.0
    },
    'Phase 2': {
        'clarity':        3.0,
        'conciseness':    2.1,
        'actionability':  3.0,
        'accuracy':       2.1,
        'cognitive_load': 2.1
    }
}

# convert to dataframe
hci_df = pd.DataFrame(hci_scores).T
hci_df['total'] = hci_df.sum(axis=1)
hci_df.index.name = 'phase'

print("HCI Rubric Scores:")
print(hci_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('HCI Rubric Evaluation — Critical Care Settings',
             fontsize=14, fontweight='bold')

# plot 1 - scores by criterion
criteria = ['clarity', 'conciseness', 'actionability', 
            'accuracy', 'cognitive_load']
x = np.arange(len(criteria))
width = 0.25

axes[0].bar(x - width, hci_df.loc['Phase 1a', criteria],
            width, label='Phase 1a', color='steelblue')
axes[0].bar(x,         hci_df.loc['Phase 1b', criteria],
            width, label='Phase 1b', color='seagreen')
axes[0].bar(x + width, hci_df.loc['Phase 2',  criteria],
            width, label='Phase 2',  color='coral')

axes[0].set_title('HCI Scores by Criterion')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Clarity', 'Conciseness',
                          'Actionability', 'Accuracy',
                          'Cognitive\nLoad'])
axes[0].set_ylabel('Average Score (1-3)')
axes[0].set_ylim(1, 3.2)
axes[0].legend()

# plot 2 - total HCI scores
phases = ['Phase 1a', 'Phase 1b', 'Phase 2']
totals = hci_df['total'].values
colors = ['steelblue', 'seagreen', 'coral']

bars = axes[1].bar(phases, totals, color=colors, width=0.5)
axes[1].set_title('Total HCI Score by Phase (max 15)')
axes[1].set_ylabel('Total HCI Score')
axes[1].set_ylim(8, 15)

# add score labels on bars
for bar, total in zip(bars, totals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                  bar.get_height() + 0.1,
                  f'{total:.1f}', ha='center',
                  fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('hci_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("HCI visualization saved!")

In [ ]:
# bring all evaluation results together in one summary table
print("="*60)
print("COMPLETE EVALUATION SUMMARY")
print("="*60)

summary_data = {
    'Evaluation Method': [
        'LLM Judge (total/12)',
        'Manual Score (total/12)',
        'ROUGE-1',
        'HCI Rubric (total/15)'
    ],
    'Phase 1a': [11.15, 8.6,  0.267, 10.1],
    'Phase 1b': [11.60, 10.8, 0.260, 11.7],
    'Phase 2':  [11.55, 11.2, 0.259, 12.3]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print("="*60)
print("\nOverall Winner by method:")
print("LLM Judge:    Phase 1b")
print("Manual Score: Phase 2")
print("ROUGE:        Inconclusive")
print("HCI Rubric:   Phase 2")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# summary data
summary_data = {
    'Evaluation Method': [
        'LLM Judge (total/12)',
        'Manual Score (total/12)',
        'ROUGE-1',
        'HCI Rubric (total/15)'
    ],
    'Phase 1a': [11.15, 8.6,  0.267, 10.1],
    'Phase 1b': [11.60, 10.8, 0.260, 11.7],
    'Phase 2':  [11.55, 11.2, 0.259, 12.3]
}

summary_df = pd.DataFrame(summary_data)

# print table
print("="*60)
print("COMPLETE EVALUATION SUMMARY")
print("="*60)
print(summary_df.to_string(index=False))
print("="*60)
print("\nOverall Winner by method:")
print("LLM Judge:    Phase 1b")
print("Manual Score: Phase 2")
print("ROUGE:        Inconclusive")
print("HCI Rubric:   Phase 2")

# visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Complete Evaluation Summary — All Three Phases',
             fontsize=15, fontweight='bold', y=1.02)

colors = ['steelblue', 'seagreen', 'coral']
phases = ['Phase 1a', 'Phase 1b', 'Phase 2']
x = np.arange(3)
width = 0.5

# plot 1 - LLM Judge vs Manual Score (normalized to /12)
ax1 = axes[0]
llm_scores   = [11.15, 11.60, 11.55]
manual_scores = [8.6,  10.8,  11.2]

bars1 = ax1.bar(x - 0.2, llm_scores,    0.35, label='LLM Judge',    color=colors, alpha=0.9)
bars2 = ax1.bar(x + 0.2, manual_scores, 0.35, label='Manual Score', color=colors, alpha=0.5)

ax1.set_title('LLM Judge vs Manual Score (/12)', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(phases)
ax1.set_ylim(7, 12.5)
ax1.set_ylabel('Score')
ax1.legend(['LLM Judge', 'Manual Score'], fontsize=9)

for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{bar.get_height():.2f}', ha='center', fontsize=9)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{bar.get_height():.1f}', ha='center', fontsize=9)

# plot 2 - ROUGE-1
ax2 = axes[1]
rouge_scores = [0.267, 0.260, 0.259]
bars3 = ax2.bar(x, rouge_scores, width, color=colors, alpha=0.9)
ax2.set_title('ROUGE-1 Scores', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(phases)
ax2.set_ylim(0.25, 0.28)
ax2.set_ylabel('Score')
ax2.axhline(y=sum(rouge_scores)/3, color='red', linestyle='--',
            label=f'Mean: {sum(rouge_scores)/3:.3f}')
ax2.legend(fontsize=9)
for bar in bars3:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
             f'{bar.get_height():.3f}', ha='center', fontsize=9)

# plot 3 - HCI Rubric
ax3 = axes[2]
hci_scores = [10.1, 11.7, 12.3]
bars4 = ax3.bar(x, hci_scores, width, color=colors, alpha=0.9)
ax3.set_title('HCI Rubric (/15)', fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(phases)
ax3.set_ylim(8, 14)
ax3.set_ylabel('Score')
for bar in bars4:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{bar.get_height():.1f}', ha='center', fontsize=9)

# shared legend
legend_patches = [
    mpatches.Patch(color='steelblue', label='Phase 1a'),
    mpatches.Patch(color='seagreen',  label='Phase 1b'),
    mpatches.Patch(color='coral',     label='Phase 2'),
]
fig.legend(handles=legend_patches, loc='lower center',
           ncol=3, fontsize=11, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig('evaluation_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Evaluation summary visualization saved!")

## 5. How Good — Evaluation Results

### Evaluation Approach
Three evaluation methods were used to assess all three phases 
on the same 20 patient questions:

**LLM-as-Judge** was used as the primary evaluation method, 
as recommended by our expert consultant. GPT-4o-mini scored 
each response on correctness, relevance, clarity and 
actionability on a scale of 1 to 3.

**Manual Scoring** was performed on the first 5 questions 
to validate the LLM judge before running it on all 20. 
Both methods broadly agreed on the direction of results.

**ROUGE Scores** were computed as a secondary metric to 
measure text overlap between chatbot responses and actual 
doctor answers.

**HCI Rubric** was applied manually to 10 responses, scoring 
each on clarity, conciseness, actionability, accuracy and 
cognitive load — criteria designed specifically for critical 
care settings.

### LLM Judge Results

| Phase | Correctness | Relevance | Clarity | Actionability | Total |
|---|---|---|---|---|---|
| Phase 1a | 2.60 | 3.00 | 3.00 | 2.55 | 11.15 |
| Phase 1b | 2.70 | 3.00 | 3.00 | 2.90 | 11.60 |
| Phase 2  | 2.60 | 3.00 | 3.00 | 2.95 | 11.55 |

Phase 1b scored highest overall at 11.60, followed closely 
by Phase 2 at 11.55. All three phases scored 3.0 on relevance 
and clarity, showing that GPT-4o-mini is consistently strong 
on these criteria regardless of the prompt or retrieval setup. 
The main differentiation between phases was on correctness 
and actionability.

### ROUGE Results

| Phase | ROUGE-1 | ROUGE-2 | ROUGE-L |
|---|---|---|---|
| Phase 1a | 0.267 | 0.036 | 0.137 |
| Phase 1b | 0.260 | 0.036 | 0.128 |
| Phase 2  | 0.259 | 0.036 | 0.133 |

ROUGE scores were consistently low and similar across all 
three phases, confirming our expert's advice that ROUGE is 
not a suitable primary metric for open-ended medical chatbot 
evaluation. The LLM tends to use different medical vocabulary 
and sentence structure compared to the dataset doctor responses, 
even when the clinical content is correct.

### HCI Rubric Results

| Criterion | Phase 1a | Phase 1b | Phase 2 |
|---|---|---|---|
| Clarity | 2.5 | 2.9 | 3.0 |
| Conciseness | 1.9 | 2.0 | 2.1 |
| Actionability | 2.1 | 2.8 | 3.0 |
| Accuracy | 1.7 | 2.0 | 2.1 |
| Cognitive Load | 1.9 | 2.0 | 2.1 |
| **Total** | **10.1** | **11.7** | **12.3** |

Phase 2 scored highest on the HCI rubric at 12.3, suggesting 
that RAG improves response suitability for critical care 
settings. Conciseness was the weakest criterion across all 
phases, indicating that all three produce responses that are 
too verbose for time-pressured clinical environments.

### Discussion
Results across all evaluation methods tell a consistent story. 
Prompt engineering alone (Phase 1b) produced a meaningful 
improvement over the true baseline (Phase 1a), showing that 
a well-crafted system prompt is a powerful lever even without 
retrieval. Adding RAG (Phase 2) further improved actionability 
and HCI suitability but did not dramatically outperform Phase 1b 
on LLM judge scores.

One interesting finding was that Phase 1b produced more specific 
medication recommendations than Phase 2 in several cases. This 
suggests that at a knowledge base size of 50,000 dialogues, 
prompt engineering has a stronger influence on response 
specificity than retrieval augmentation. A larger knowledge 
base or re-ranking strategies could potentially address this 
in future work.

The consistent weakness in conciseness across all phases is 
the most important HCI finding. None of the three phases 
produce responses that are brief enough for critical care 
settings without further refinement.

## 6. Where — Deployment and Interface

### Current Deployment
This project runs as a self-contained Jupyter Notebook in VS Code. 
All components including the FAISS index, embeddings and chatbot 
functions are contained within the notebook, making it easy to 
run and reproduce.

The following files are saved locally alongside the notebook:
- doctor_embeddings.npy — embedded knowledge base
- medical_faiss.index — FAISS vector index
- test_set_responses.csv — 20 test questions and responses
- judge_results.csv — LLM judge evaluation results

### Chat Interface
A simple chat interface was built using ipywidgets directly inside 
the notebook. ipywidgets lets you create interactive elements like 
text boxes and buttons inside Jupyter without needing a separate 
web app. Users can type a patient question and get a Phase 2 RAG 
response without leaving the notebook.

### Future Deployment
A production version could be deployed as a web app using Streamlit 
or Gradio, hosted free on Hugging Face Spaces or Streamlit Cloud. 
This would include a larger knowledge base, clinical safety filtering 
and an interface designed for critical care settings.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# create the interface components
title = widgets.HTML(
    value="<h3 style='color:#1F4E79'>Medical Chatbot with RAG</h3>"
          "<p style='color:gray'>Powered by GPT-4o-mini and HealthCareMagic knowledge base</p>"
)

question_input = widgets.Textarea(
    placeholder="Type your medical question here...",
    layout=widgets.Layout(width='100%', height='80px')
)

submit_button = widgets.Button(
    description="Ask Doctor",
    button_style='primary',
    layout=widgets.Layout(width='150px', height='35px')
)

output_area = widgets.Output()

# what happens when button is clicked
def on_submit(button):
    with output_area:
        clear_output()
        
        question = question_input.value.strip()
        
        if not question:
            print("Please type a question first.")
            return
        
        print("Patient Question:")
        print(question)
        print("\nRetrieving similar cases...")
        
        # retrieve similar cases
        similar_cases = retrieve_similar_cases(question, top_k=3)
        print(f"Found {len(similar_cases)} similar cases from knowledge base.")
        
        print("\nGenerating response...")
        response = phase2_rag_chatbot(question)
        
        print("\n" + "="*60)
        print("Doctor Response (RAG):")
        print("="*60)
        print(response)
        print("="*60)
        
        # show retrieved cases
        print("\nRetrieved Cases Used as Context:")
        for i, case in enumerate(similar_cases, 1):
            print(f"\nCase {i}: {case['patient_question'][:100]}...")

submit_button.on_click(on_submit)

# display everything
display(title)
display(question_input)
display(submit_button)
display(output_area)

## 7. Experiences — Reflection and Lessons Learned

### What Worked Well
The three-phase approach worked really well. Having a baseline 
early gave a clear reference point and made the RAG comparison 
meaningful. Building the pipeline manually using 
sentence-transformers, FAISS and the OpenAI API kept things 
transparent and easy to debug.

The HealthCareMagic dataset was a great choice — clean, minimal 
preprocessing needed and wide topic coverage. Consulting an LLM 
expert before starting also saved significant time and led to 
better technical decisions around model selection, index type 
and evaluation approach.

### What Did Not Work as Expected
Prompt engineering alone (Phase 1b) produced more specific 
medication recommendations than the full RAG system (Phase 2) 
in several cases. This was unexpected and suggests that at 
50,000 dialogues, a well-crafted prompt has more influence 
on specificity than retrieval augmentation alone.

ROUGE scores were also inconclusive across all phases, 
confirming it is not a suitable metric for open-ended medical 
chatbot evaluation.

### Limitations
- Only 20 test questions were used, which is a small sample
- The HCI rubric was applied by a single non-clinical evaluator
- All three phases produced responses too verbose for critical 
  care settings
- The system has not been tested with actual clinicians

### Future Work
- Expand knowledge base to full 112,165 dialogues
- Add response length constraints to address verbosity
- Conduct user testing with clinicians in critical care settings
- Explore biomedical-specific embedding models
- Deploy as a web application using Streamlit or Gradio

### Personal Reflection
This project was a valuable learning experience in applied AI. 
Coming in with limited LLM experience, building a working RAG 
system from scratch within a tight timeline was challenging 
but very rewarding.

The most interesting insight was how much a well-crafted system 
prompt alone can improve LLM responses, sometimes more than 
adding a full retrieval system. This connects directly to my 
PhD research in HCI and healthcare technologies, particularly 
around designing and evaluating AI tools for critical care 
settings.

Beyond the chatbot itself, I hope the HCI evaluation framework 
I developed here could be useful for future research on designing 
and assessing AI tools for clinical settings. In practice a system 
like this could help clinicians access quick medical reference 
information faster and support medical students during training.

## Testing out Gradio deployment

In [ ]:
import subprocess
subprocess.run(["pip", "install", "gradio"])
print("Gradio installed!")

In [ ]:
import gradio as gr

def medical_chatbot(message, history):
    response = phase2_rag_chatbot(message)
    return response

demo = gr.ChatInterface(
    fn=medical_chatbot,
    title="Medical Chatbot with RAG",
    description="Powered by GPT-4o-mini and HealthCareMagic 50k knowledge base",
    examples=[
        "I have chest pain and shortness of breath",
        "I had knee replacement surgery 6 weeks ago and still have swelling",
        "I accidentally took double my blood pressure medication"
    ]
)

demo.launch(share=True)

In [ ]:
# save knowledge base to CSV for Hugging Face deployment
knowledge_base = df.head(50000).reset_index(drop=True)
knowledge_base.to_csv("knowledge_base.csv", index=False)
print(f"Knowledge base saved!")
print(f"Rows: {len(knowledge_base)}")
print(f"Columns: {knowledge_base.columns.tolist()}")